# Taylor Root Prediction — Reviewer Demo Notebook (v5)

This notebook is designed so a reviewer can **clone the repo and run everything with relative paths**, even when the notebook is executed from `notebooks/`.

What you can do here:
- (Optional) generate **root-regression** and **interval** NPZ datasets under `data/`
- (Optional) train models (ANN / LSTM / Anchored-MLP / Transformer interval predictor)
- run evaluation (K-sweep + baseline) and produce plots/reports

> Tip: start small (e.g., 20k samples) to verify reproducibility quickly.


In [2]:
from __future__ import annotations

from pathlib import Path
import os, sys, subprocess, shutil, textwrap
import json
import numpy as np

def find_repo_root(start: Path | None = None) -> Path:
    """Find repo root by walking up until we see required folders."""
    if start is None:
        start = Path.cwd().resolve()
    else:
        start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "configs").is_dir() and (p / "models").is_dir():
            return p
    raise RuntimeError("Could not find repo root (expected folders: configs/, models/).")

REPO = find_repo_root()
print("CWD :", Path.cwd())
print("REPO:", REPO)

# Make all relative paths resolve from repo root
os.chdir(REPO)
print("CHDIR ->", Path.cwd())

def R(rel: str | Path) -> Path:
    return (REPO / Path(rel)).resolve()

def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)

def run(cmd, env=None, cwd=None):
    """Run a command, printing stdout/stderr. Raises on failure."""
    cmd = [str(x) for x in cmd]
    print(">>", " ".join(cmd))
    r = subprocess.run(cmd, cwd=str(cwd or REPO), env=env, capture_output=True, text=True)
    if r.stdout:
        print(r.stdout)
    if r.returncode != 0:
        if r.stderr:
            print("----- STDERR -----")
            print(r.stderr)
        raise RuntimeError(f"Command failed (code={r.returncode})")
    return r

# Ensure imports like `import src...` work when scripts are executed via subprocess
BASE_ENV = os.environ.copy()
BASE_ENV["PYTHONPATH"] = str(REPO)


CWD : /home/seokjun/taylor-root-prediction/notebooks
REPO: /home/seokjun/taylor-root-prediction
CHDIR -> /home/seokjun/taylor-root-prediction


## 0) Dependencies

Minimum:
- numpy, pyyaml, tqdm
- torch (CPU or CUDA)
- matplotlib (for plots in evaluation)

Optional (only if you enable symbolic baselines in evaluation):
- sympy
- requests


In [3]:
# (Optional) quick sanity check
import importlib, platform

pkgs = ["numpy","yaml","tqdm","torch","matplotlib"]
missing = []
for p in pkgs:
    try:
        importlib.import_module(p if p!="yaml" else "yaml")
    except Exception:
        missing.append(p)
print("Missing:", missing)

import torch
print("Python :", platform.python_version())
print("Torch  :", torch.__version__)
print("CUDA?  :", torch.cuda.is_available())


Missing: []
Python : 3.10.18
Torch  : 2.5.1+cu121
CUDA?  : True


## 1) Configure a small reviewer run

These defaults are intentionally small so the notebook finishes in reasonable time.
Increase sizes/epochs if you want stronger models.


In [4]:
DEGREE = 25

# Recommended quick sizes for reviewers
N_TOTAL_ROOT = 20000
N_TOTAL_INTERVAL = 20000

# Output directories (ALWAYS under repo/data/)
OUT_ROOT_DIR = R("data/taylor_data_physchem_v4_deg25")
OUT_INTERVAL_DIR = R("data/taylor_data_physchem_v4_interval")
ensure_dir(OUT_ROOT_DIR)
ensure_dir(OUT_INTERVAL_DIR)

print("OUT_ROOT_DIR    :", OUT_ROOT_DIR)
print("OUT_INTERVAL_DIR:", OUT_INTERVAL_DIR)

train_npz = OUT_ROOT_DIR / f"taylor_deg{DEGREE}_train.npz"
val_npz   = OUT_ROOT_DIR / f"taylor_deg{DEGREE}_val.npz"
test_npz  = OUT_ROOT_DIR / f"taylor_deg{DEGREE}_test.npz"

itr_train_npz = OUT_INTERVAL_DIR / f"taylor_deg{DEGREE}_train.npz"
itr_val_npz   = OUT_INTERVAL_DIR / f"taylor_deg{DEGREE}_val.npz"
itr_test_npz  = OUT_INTERVAL_DIR / f"taylor_deg{DEGREE}_test.npz"


OUT_ROOT_DIR    : /home/seokjun/taylor-root-prediction/data/taylor_data_physchem_v4_deg25
OUT_INTERVAL_DIR: /home/seokjun/taylor-root-prediction/data/taylor_data_physchem_v4_interval


## 2) (Optional) Fix “dataset created at repo root” issue

If you previously generated datasets but they ended up in:
- `<repo>/taylor_data_physchem_v4_deg25/`
- `<repo>/taylor_data_physchem_v4_interval/`

this cell will move them into `data/` automatically.


In [5]:
def migrate_dataset_if_found(folder_name: str):
    wrong = REPO / folder_name
    correct = REPO / "data" / folder_name
    if wrong.exists() and wrong.is_dir():
        ensure_dir(correct.parent)
        if correct.exists():
            # If target already populated, keep it.
            if any(correct.iterdir()):
                print("[MIGRATE] target already populated:", correct)
                return
            else:
                shutil.rmtree(correct)
        print("[MIGRATE] moving", wrong, "->", correct)
        shutil.move(str(wrong), str(correct))

migrate_dataset_if_found("taylor_data_physchem_v4_deg25")
migrate_dataset_if_found("taylor_data_physchem_v4_interval")

print("[CHECK]")
print(" root dir exists   :", OUT_ROOT_DIR.exists())
print(" interval dir exists:", OUT_INTERVAL_DIR.exists())


[MIGRATE] target already populated: /home/seokjun/taylor-root-prediction/data/taylor_data_physchem_v4_deg25
[CHECK]
 root dir exists   : True
 interval dir exists: True


## 3) Generate datasets (NPZ) under `data/`

This notebook assumes your repo includes dataset generator scripts (recommended):
- `data_generation/generate_root_dataset_v4.py`  (root-regression NPZ)
- `data_generation/generate_interval_dataset_v4.py` (interval NPZ)

If your filenames differ, edit the paths below **once**, and the rest of the notebook will work.


In [6]:
# ---- generator script locations (edit if your repo uses different names) ----
gen_root = R("scripts/data/generate_dataset_physchem_v4.py")
gen_interval = R("scripts/data/generate_interval_dataset_physchem_v4.py")

print("gen_root    :", gen_root, "exists=", gen_root.exists())
print("gen_interval:", gen_interval, "exists=", gen_interval.exists())

# If scripts are missing, try to auto-discover by keyword
if not gen_root.exists():
    cands = list(REPO.rglob("*root*dataset*gen*.py")) + list(REPO.rglob("*generate*root*.py"))
    cands = [p for p in cands if p.is_file()]
    if cands:
        gen_root = cands[0]
        print("[AUTO] gen_root ->", gen_root)
if not gen_interval.exists():
    cands = list(REPO.rglob("*interval*dataset*gen*.py")) + list(REPO.rglob("*generate*interval*.py"))
    cands = [p for p in cands if p.is_file()]
    if cands:
        gen_interval = cands[0]
        print("[AUTO] gen_interval ->", gen_interval)

assert gen_root.exists(), "Root dataset generator script not found in repo."
assert gen_interval.exists(), "Interval dataset generator script not found in repo."

# ---- root regression dataset ----
need_root = (not train_npz.exists()) or (not val_npz.exists()) or (not test_npz.exists())
if need_root:
    run([
        sys.executable, str(gen_root),
        "--degree", str(DEGREE),
        "--n-total", str(N_TOTAL_ROOT),
        "--seed", "42",
        "--out-dir", str(OUT_ROOT_DIR),   # ABSOLUTE PATH to avoid CWD issues
        "--save-expr-str", "1",
    ], env=BASE_ENV, cwd=REPO)
else:
    print("[SKIP] root dataset already exists.")

# ---- interval dataset ----
need_interval = (not itr_train_npz.exists()) or (not itr_val_npz.exists()) or (not itr_test_npz.exists())
if need_interval:
    run([
        sys.executable, str(gen_interval),
        "--degree", str(DEGREE),
        "--n-total", str(N_TOTAL_INTERVAL),
        "--seed", "42",
        "--out-dir", str(OUT_INTERVAL_DIR),  # ABSOLUTE PATH
        "--save-expr-str", "1",
    ], env=BASE_ENV, cwd=REPO)
else:
    print("[SKIP] interval dataset already exists.")

print("[DATA CHECK]")
for p in [train_npz, val_npz, test_npz, itr_train_npz, itr_val_npz, itr_test_npz]:
    print(" ", p.relative_to(REPO), "exists=", p.exists())


gen_root    : /home/seokjun/taylor-root-prediction/scripts/data/generate_dataset_physchem_v4.py exists= True
gen_interval: /home/seokjun/taylor-root-prediction/scripts/data/generate_interval_dataset_physchem_v4.py exists= True
[SKIP] root dataset already exists.
[SKIP] interval dataset already exists.
[DATA CHECK]
  data/taylor_data_physchem_v4_deg25/taylor_deg25_train.npz exists= True
  data/taylor_data_physchem_v4_deg25/taylor_deg25_val.npz exists= True
  data/taylor_data_physchem_v4_deg25/taylor_deg25_test.npz exists= True
  data/taylor_data_physchem_v4_interval/taylor_deg25_train.npz exists= True
  data/taylor_data_physchem_v4_interval/taylor_deg25_val.npz exists= True
  data/taylor_data_physchem_v4_interval/taylor_deg25_test.npz exists= True


## 4) (Optional) Train models (YAML-driven)

All training scripts use environment variables (no CLI args needed):
- `TAYLOR_CFG` (or `CFG_PATH` for transformer)
- `TRAIN_NPZ`, `VAL_NPZ`, `TEST_NPZ`
- `OUT_DIR`
- `DEVICE`

By default, training cells are **commented out** to keep reviewer runtime reasonable.
Uncomment what you want to run.


In [11]:
import torch

def train_script(script_rel: str, cfg_rel: str, out_rel: str, train_path: Path, val_path: Path, test_path: Path | None):
    script = R(script_rel)
    cfg = R(cfg_rel)
    out_dir = R(out_rel)
    ensure_dir(out_dir)

    env = BASE_ENV.copy()
    env.update({
        "TAYLOR_CFG": str(cfg),
        "TRAIN_NPZ": str(train_path),
        "VAL_NPZ": str(val_path),
        "TEST_NPZ": (str(test_path) if test_path is not None else ""),
        "OUT_DIR": str(out_dir),
        "DEVICE": ("cuda" if torch.cuda.is_available() else "cpu"),
    })
    run([sys.executable, str(script)], env=env, cwd=REPO)

# ---- Uncomment the models you want to train ----

train_script("models/taylor_nn/ann.py",  "configs/taylor_root_ann.yaml",  "results/taylor_nn/ann",
             train_npz, val_npz, test_npz)

train_script("models/taylor_nn/lstm.py", "configs/taylor_root_lstm.yaml", "results/taylor_nn/lstm",
             train_npz, val_npz, test_npz)

train_script("models/taylor_nn/mlp.py",  "configs/taylor_root_mlp.yaml",  "results/taylor_nn/mlp",
             train_npz, val_npz, test_npz)


>> /home/seokjun/miniconda3/envs/action_recognition/bin/python /home/seokjun/taylor-root-prediction/models/taylor_nn/ann.py
[CONFIG] /home/seokjun/taylor-root-prediction/configs/taylor_root_ann.yaml
[DATA] train=800000 val=100000 test=100000
[NPZ] train coeff_key=coeffs, keys=['coeffs', 'root0', 'root1', 'root2', 'func_id', 'degree', 'template_str', 'norm_scale', 'root_count', 'roots', 'expr_str']
[SHAPE] coeff_dim(D)=26, num_roots=25
[MODEL] hidden_dim=25, layers=auto, act=tanh
[OUT] /home/seokjun/taylor-root-prediction/results/taylor_nn/ann
[ep=   1] train_loss=0.0996695  val_loss=0.085333
  -> save best: /home/seokjun/taylor-root-prediction/results/taylor_nn/ann/best.pt
[ep=   2] train_loss=0.0755342  val_loss=0.067131
  -> save best: /home/seokjun/taylor-root-prediction/results/taylor_nn/ann/best.pt
[ep=   3] train_loss=0.0619915  val_loss=0.058033
  -> save best: /home/seokjun/taylor-root-prediction/results/taylor_nn/ann/best.pt
[ep=   4] train_loss=0.0557657  val_loss=0.0538373
 

## 5) (Optional) Train Transformer interval predictor

The transformer script reads:
- `CFG_PATH`
- `TRAIN_NPZ`, `VAL_NPZ`, `TEST_NPZ`
- `OUT_DIR`, `DEVICE`, `MODE=train`

Uncomment to train.


In [12]:
def train_transformer():
    script = R("models/transformer/model.py")
    cfg = R("configs/transformer_interval.yaml")
    out_dir = R("results/transformer_interval")
    ensure_dir(out_dir)

    env = BASE_ENV.copy()
    env.update({
        "CFG_PATH": str(cfg),
        "TRAIN_NPZ": str(itr_train_npz),
        "VAL_NPZ": str(itr_val_npz),
        "TEST_NPZ": str(itr_test_npz),
        "OUT_DIR": str(out_dir),
        "DEVICE": ("cuda" if torch.cuda.is_available() else "cpu"),
        "MODE": "train",
    })
    run([sys.executable, str(script)], env=env, cwd=REPO)

# Uncomment to train transformer
train_transformer()


>> /home/seokjun/miniconda3/envs/action_recognition/bin/python /home/seokjun/taylor-root-prediction/models/transformer/model.py
[CFG]  /home/seokjun/taylor-root-prediction/configs/transformer_interval.yaml
[DATA] train=800000 val=100000 test=100000
[VOCAB] size=22  max_len=16 (rule=auto_top_1_percent_mean_x1_1)  top_k=25
[ARCH] layers=4 heads=8 hidden_dim=256
[SCALE] asinh/sinh scale=1
[OUT]  /home/seokjun/taylor-root-prediction/results/transformer_interval
[VAL] ep=001 min_mae=0.464149 min_p90=1.34287 max_mae=0.4813 max_p90=1.36013 n=100000
[SAVE] best -> /home/seokjun/taylor-root-prediction/results/transformer_interval/best.pt (best_val_max_mae=0.4813)
[VAL] ep=002 min_mae=0.457572 min_p90=1.33213 max_mae=0.488043 max_p90=1.36293 n=100000
[VAL] ep=003 min_mae=0.466148 min_p90=1.33966 max_mae=0.479412 max_p90=1.35299 n=100000
[SAVE] best -> /home/seokjun/taylor-root-prediction/results/transformer_interval/best.pt (best_val_max_mae=0.479412)
[VAL] ep=004 min_mae=0.465269 min_p90=1.3405

## 6) Make Test Dataset

In [ ]:
# =========================
# (NEW) Build test dataset before evaluation
# =========================
from pathlib import Path
import os, sys, subprocess, textwrap

# ---- repo root 추정 (notebooks/ 에서 실행해도 안전) ----
CWD = Path.cwd().resolve()
REPO = CWD if (CWD / "configs").exists() else CWD.parent
assert (REPO / "configs").exists(), f"Repo root not found. CWD={CWD}"

print("CWD :", CWD)
print("REPO:", REPO)

def run(cmd, env=None, cwd=None):
    cmd = [str(x) for x in cmd]
    print("\n>>", " ".join(cmd))
    r = subprocess.run(cmd, cwd=str(cwd) if cwd else None, env=env, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if r.stdout:
        print(r.stdout)
    if r.returncode != 0:
        print("----- STDERR -----")
        print(r.stderr)
        raise RuntimeError(f"Command failed (code={r.returncode})")
    return r

# ---- 설정 ----
FORCE_REGEN = False  # True로 바꾸면 항상 새로 생성(덮어쓰기)

OUT_NPZ = REPO / "v6/taylor_test_physchem_v3_allroots_10000.npz"
CFG_YAML = REPO / "configs/make_test_dataset.yaml"
SCRIPT  = REPO / "scripts/data/make_test_dataset.py"  # 우리가 권장한 위치

# ---- sanity check ----
assert SCRIPT.exists(), f"Missing generator script: {SCRIPT}"
assert CFG_YAML.exists(), f"Missing yaml config: {CFG_YAML}"

OUT_NPZ.parent.mkdir(parents=True, exist_ok=True)

if OUT_NPZ.exists() and not FORCE_REGEN:
    print(f"[SKIP] Test dataset already exists: {OUT_NPZ.relative_to(REPO)} ({OUT_NPZ.stat().st_size/1e6:.2f} MB)")
else:
    if OUT_NPZ.exists() and FORCE_REGEN:
        OUT_NPZ.unlink()

    env = dict(os.environ)
    env["PYTHONPATH"] = str(REPO)
    env["CFG_PATH"] = str(CFG_YAML)     # ✅ make_test_dataset.py가 이 환경변수로 yaml 읽게 만들었을 때 기준

    print(f"[GEN] Creating test dataset -> {OUT_NPZ.relative_to(REPO)}")
    run([sys.executable, str(SCRIPT)], env=env, cwd=REPO)

    if not OUT_NPZ.exists():
        raise FileNotFoundError(f"Generation finished but output not found: {OUT_NPZ}")

print("[OK] Test dataset ready:", OUT_NPZ.relative_to(REPO))


CWD : /home/seokjun/taylor-root-prediction
REPO: /home/seokjun/taylor-root-prediction
[GEN] Creating test dataset -> data/test/v6/taylor_test_physchem_v3_allroots_10000.npz

>> /home/seokjun/miniconda3/envs/action_recognition/bin/python /home/seokjun/taylor-root-prediction/scripts/data/make_test_dataset.py
[CFG] /home/seokjun/taylor-root-prediction/configs/make_test_dataset.yaml
[REPO] /home/seokjun/taylor-root-prediction
[OUT] /home/seokjun/taylor-root-prediction/data/test/v6/taylor_test_physchem_v3_allroots_10000.npz
[INFO] templates:
   0: damped_osc_1       role=time
   1: damped_osc_2       role=time
   2: double_well        role=real
   3: spring_tanh        role=real
   4: rc_step            role=time
   5: damped_sine_current role=time
   6: heat_fin           role=time
   7: boltzmann_prob     role=real
   8: ising_magnet       role=real
   9: activation_log     role=concentration
  10: arrhenius          role=temperature_K
  11: diode_iv           role=real
  12: damped_wave 

## 7) Evaluation (K-sweep + baseline + reports/plots)

This uses a YAML config file and writes everything to `results/`.
You need trained checkpoints for the models you want to include.

Uncomment to run evaluation after training.


In [ ]:
# [ipynb] run_evaluation() + 로그파일 저장 + less 보기까지 합친 버전

import os, sys, subprocess
from pathlib import Path

def run_evaluation(save_log: bool = True, open_less: bool = True, log_name: str = "eval_k_sweep_full.log"):
    script = R("scripts/eval/evaluate_k_sweep.py")
    cfg = R("configs/eval_k_sweep.yaml")
    outdir = R("results/runs_k_sweep_viz")
    ensure_dir(outdir)

    env = BASE_ENV.copy()
    env.update({
        "EVAL_CFG": str(cfg),
        "OUTDIR": str(outdir),
        "DEVICE": ("cuda" if torch.cuda.is_available() else "cpu"),
    })

    if save_log:
        log_dir = R("results/logs")
        ensure_dir(log_dir)
        log_path = Path(log_dir) / log_name

        # stdout+stderr를 파일로 저장
        with open(log_path, "w", encoding="utf-8") as f:
            p = subprocess.Popen(
                [sys.executable, str(script)],
                cwd=str(REPO),
                env=env,
                stdout=f,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )
            ret = p.wait()

        print("return_code =", ret)
        print("log_saved_to =", log_path)

        # less로 열기 (q 종료, /검색 Enter, n 다음)
        if open_less:
            subprocess.run(["bash", "-lc", f"less -R {str(log_path)}"], cwd=str(REPO))

        return ret, log_path

    # 로그 저장 안 하면 기존처럼 바로 실행
    run([sys.executable, str(script)], env=env, cwd=REPO)
    return 0, None

# 실행
run_evaluation(save_log=True, open_less=True)


## 8) Quick load of “fail-by-func_id” CSV (if you enabled the report)

If evaluation produced `fail_by_funcid_*.csv`, you can inspect it here.


In [8]:
try:
    import pandas as pd
    from IPython.display import display
except Exception:
    pd = None

outdir = R("results/runs_k_sweep_viz")
cands = sorted(outdir.glob("fail_by_funcid_*.csv"))
print("Found:", [p.relative_to(REPO) for p in cands])

if pd is not None and cands:
    df = pd.read_csv(cands[0])
    display(df.head(30))
else:
    print("[INFO] No CSV found yet (run evaluation with report_fail_funcid enabled).")


Found: []
[INFO] No CSV found yet (run evaluation with report_fail_funcid enabled).
